In [4]:
import pandas as pd

import numpy as np

import json



# ==========================================

# 1. DATA LOADING & ROBUST CLEANING

# ==========================================

print("--- Step 1: Loading and Cleaning Data ---")



# Load Data

df = pd.read_csv('Results.csv')



# Fix Typo in Column Names

df = df.rename(columns={

    'delta_x_axis_acutal_position': 'delta_x_axis_actual_position',

    'x_axis_acutal_position': 'x_axis_actual_position'

})



# Calculate 'delta_timestamp'

df = df.sort_values(by='timestamp')

df['delta_timestamp'] = df['timestamp'].diff().fillna(0)



# Clean string columns (remove quotes)

cols_to_clean = [

    'delta_mic', 'delta_x_axis_actual_position', 'delta_y_axis_actual_position',

    'delta_z_axis_actual_position', 'x_axis_actual_position',

    'y_axis_actual_position', 'z_axis_actual_position'

]



for col in cols_to_clean:

    if col in df.columns:

        df[col] = df[col].astype(str).str.replace("'", "", regex=False)

        df[col] = pd.to_numeric(df[col], errors='coerce')



df_clean = df.dropna().copy()



# Labeling Logic

def get_machine_status(ts):

    if ts <= 1764623236.41566: return 'Idle'

    elif ts <= 1764623527.88477: return 'Cutting'

    elif ts <= 1764623680.71636: return 'Idle'

    elif ts <= 1764623823.94084: return 'Traveling'

    else: return 'Idle'



df_clean['Machine_status'] = df_clean['timestamp'].apply(get_machine_status)

df_clean['target'] = df_clean['Machine_status'].map({'Idle': 0, 'Cutting': 1, 'Traveling': 2})



# Features to use

features = [

    'delta_mic', 'delta_timestamp', 'delta_x_axis_actual_position',

    'delta_y_axis_actual_position', 'delta_z_axis_actual_position',

    'mic', 'x_axis_actual_position', 'y_axis_actual_position', 'z_axis_actual_position'

]



X = df_clean[features]

y = df_clean['target']



print(f"Data ready. Shape: {df_clean.shape}")



# ==========================================

# 2. CUSTOM MODEL CLASSES (Training)

# ==========================================

class DecisionTreeClassifier:

    def __init__(self, max_depth=None):

        self.max_depth = max_depth



    def fit(self, X, y):

        self.classes_ = np.unique(y)

        self.tree_ = self._build_tree(X, y)



    def _gini(self, y):

        m = len(y)

        return 1.0 - sum((np.sum(y == c) / m) ** 2 for c in self.classes_)



    def _best_split(self, X, y):

        m, n = X.shape

        if m <= 1: return None, None

        best_gini = self._gini(y)

        best_idx, best_thr = None, None



        # Speed optimization: Randomly select features to check

        n_select = min(n, int(np.sqrt(n)))

        if n_select < 1: n_select = 1

        check_features = np.random.choice(range(n), n_select, replace=False)



        for idx in check_features:

            thresholds = np.unique(X.iloc[:, idx])

            if len(thresholds) > 50:

                thresholds = np.percentile(thresholds, np.linspace(0, 100, 50))

            for thr in thresholds:

                left_mask = X.iloc[:, idx] < thr

                y_left, y_right = y[left_mask], y[~left_mask]

                if len(y_left) == 0 or len(y_right) == 0: continue



                gini = (len(y_left)*self._gini(y_left) + len(y_right)*self._gini(y_right)) / m

                if gini < best_gini:

                    best_gini = gini

                    best_idx = idx

                    best_thr = thr

        return best_idx, best_thr



    def _build_tree(self, X, y, depth=0):

        mode_val = y.mode()

        leaf_class = int(mode_val[0]) if not mode_val.empty else 0

        node = {'type': 'leaf', 'class': leaf_class}



        if self.max_depth is None or depth < self.max_depth:

            idx, thr = self._best_split(X, y)

            if idx is not None:

                left_mask = X.iloc[:, idx] < thr

                node = {

                    'type': 'node',

                    'feature_index': int(idx),

                    'threshold': float(thr),

                    'left': self._build_tree(X[left_mask], y[left_mask], depth + 1),

                    'right': self._build_tree(X[~left_mask], y[~left_mask], depth + 1)

                }

        return node



    def to_dict(self):

        return {'tree': self.tree_}



class RandomForestClassifier:

    def __init__(self, n_estimators=10, max_depth=10):

        self.n_estimators = n_estimators

        self.max_depth = max_depth

        self.trees = []



    def fit(self, X, y):

        self.features = X.columns.tolist() # Save feature names for Lambda

        for i in range(self.n_estimators):

            print(f"Training tree {i+1}/{self.n_estimators}...")

            # Bootstrap sample

            sample_idx = np.random.choice(len(X), len(X), replace=True)

            X_sample, y_sample = X.iloc[sample_idx], y.iloc[sample_idx]



            tree = DecisionTreeClassifier(max_depth=self.max_depth)

            tree.fit(X_sample, y_sample)

            self.trees.append(tree)



    def to_dict(self):

        return {

            'n_estimators': self.n_estimators,

            'max_depth': self.max_depth,

            'features': self.features, # Crucial for mapping JSON inputs

            'trees': [t.to_dict() for t in self.trees]

        }



# ==========================================

# 3. TRAIN & EXPORT

# ==========================================

print("\n--- Training Model ---")

# Keep n_estimators low (e.g., 5-10) for lightweight Lambda execution

rf = RandomForestClassifier(n_estimators=10, max_depth=8)

rf.fit(X, y)



# Save to JSON

with open('cnc_model.json', 'w') as f:

    json.dump(rf.to_dict(), f)



print("\nSuccess! 'cnc_model.json' saved.")

# ==========================================
# 4. EVALUATION (Confusion Matrix & Accuracy)
# ==========================================
print("--- Step 4: Evaluating Model ---")

def predict_single_row(node, row_values):
    """Recursive function to traverse the tree dict"""
    if node['type'] == 'leaf':
        return node['class']

    # Compare feature value against threshold
    if row_values[node['feature_index']] < node['threshold']:
        return predict_single_row(node['left'], row_values)
    else:
        return predict_single_row(node['right'], row_values)

# Generate predictions
y_pred = []
X_values = X.values  # Convert to numpy array for speed

print(f"Generating predictions for {len(X)} samples...")

for i in range(len(X_values)):
    row = X_values[i]
    votes = []

    # Get vote from each tree in the random forest
    for tree in rf.trees:
        vote = predict_single_row(tree.tree_, row)
        votes.append(vote)

    # Majority vote (Mode)
    final_pred = max(set(votes), key=votes.count)
    y_pred.append(final_pred)

# Calculate Accuracy
y_true = y.values
accuracy = np.mean(y_true == y_pred) * 100
print(f"\nModel Accuracy: {accuracy:.2f}%")

# Generate Confusion Matrix using Pandas
confusion_matrix = pd.crosstab(
    pd.Series(y_true, name='Actual'),
    pd.Series(y_pred, name='Predicted')
)

print("\nConfusion Matrix:")
print(confusion_matrix)

--- Step 1: Loading and Cleaning Data ---
Data ready. Shape: (397, 24)

--- Training Model ---
Training tree 1/10...
Training tree 2/10...
Training tree 3/10...
Training tree 4/10...
Training tree 5/10...
Training tree 6/10...
Training tree 7/10...
Training tree 8/10...
Training tree 9/10...
Training tree 10/10...

Success! 'cnc_model.json' saved.
--- Step 4: Evaluating Model ---
Generating predictions for 397 samples...

Model Accuracy: 96.47%

Confusion Matrix:
Predicted    0    1   2
Actual                 
0          196    4   1
1            3  133   0
2            2    4  54
